In [1]:
from typing import Any, Dict

import hydra
import numpy as np
import omegaconf
import torch
import pytorch_lightning as pl
import torch.nn as nn
from torch.nn import functional as F
from torch_scatter import scatter
from tqdm import tqdm

from cdvae.common.utils import PROJECT_ROOT
from cdvae.common.data_utils import (
    EPSILON, cart_to_frac_coords, mard, lengths_angles_to_volume,
    frac_to_cart_coords, min_distance_sqr_pbc)
from cdvae.pl_modules.embeddings import MAX_ATOMIC_NUM
from cdvae.pl_modules.embeddings import KHOT_EMBEDDINGS

#added by Tsach
from pymatgen.core.structure import Structure
from pymatgen.core.periodic_table import Element
from pymatgen.core.lattice import Lattice
from pymatgen.analysis.diffraction.xrd import XRDCalculator
#import Batch
from torch_geometric.data import Batch
xrd_calculator = XRDCalculator(wavelength='CuKa', symprec=0.1)
torch.set_printoptions(threshold=50000) # use this if you want to print the entire tensor


In [2]:
 import time
import argparse
import torch

from tqdm import tqdm
from torch.optim import Adam
from pathlib import Path
from types import SimpleNamespace
from torch_geometric.data import Batch
from torch_geometric.data.separate import separate

#import a library that allows you to reload a module
from importlib import reload

from eval_utils import load_model

all_frac_coords_stack = []
all_atom_types_stack = []
frac_coords = []
num_atoms = []
atom_types = []
lengths = []
angles = []
input_data_list = []

#my code 
list_of_idxs = []
list_of_batchs = []

In [3]:
import importlib 
import eval_utils

In [4]:
importlib.reload(eval_utils)

<module 'eval_utils' from '/home/gridsan/tmackey/cdvae/scripts/eval_utils.py'>

In [5]:
from eval_utils import load_model

In [6]:
model_path = Path("/home/gridsan/tmackey/hydra/singlerun/2023-10-24/xrd_no_encoder_peak_intensities_concat_atomic_comp_concat_v2")
model, test_loader, cfg = load_model(model_path, True)

loader = test_loader

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/hydra/experimental/compose.py:16: UserWarning: hydra.experimental.compose() is no longer experimental. Use hydra.compose()
  warnings.warn(


  0%|          | 0/3785 [00:00<?, ?it/s]

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/pymatgen/io/cif.py:1168: UserWarning: Issues encountered while parsing CIF: Some fractional coordinates rounded to ideal values to avoid issues with finite precision.
  warnings.warn("Issues encountered while parsing CIF: " + "\n".join(self.warnings))


CrystDataModule(self.datasets={'train': {'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy train', 'path': '/home/gridsan/tmackey/cdvae/data/perov_5/train.csv', 'prop': 'heat_ref', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}, 'val': [{'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy val', 'path': '/home/gridsan/tmackey/cdvae/data/perov_5/val.csv', 'prop': 'heat_ref', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}], 'test': [{'_target_': 'cdvae.pl_data.dataset.CrystDataset', 'name': 'Formation energy test', 'path': '/home/gridsan/tmackey/cdvae/data/perov_5/test.csv', 'prop': 'heat_ref', 'niggli': True, 'primitive': False, 'graph_method': 'crystalnn', 'lattice_scale_method': 'scale_length', 'preprocess_workers': 30}]}, self.num_workers={'train': 0, 'val': 0, 'test': 0

/home/gridsan/tmackey/.conda/envs/cdvae/lib/python3.8/site-packages/torch_geometric/deprecation.py:13: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [9]:
for idx, batch in enumerate(loader):
    print(idx)
    list_of_idxs.append(idx)
    list_of_batchs.append(batch)

idx = list_of_idxs[0]
batch = list_of_batchs[0]

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14


In [10]:
def new_dataloader_batch_processor(batch): 
    batch_reserve = batch
    xrd_int = batch_reserve[1]
    xrd_loc = batch_reserve[2]
    atom_spec = batch_reserve[3]
    batch = batch[0]
    return batch_reserve, xrd_int, xrd_loc, atom_spec, batch

In [11]:
batch_reserve, xrd_int, xrd_loc, atom_spec, batch = new_dataloader_batch_processor(batch)

In [12]:
batch_all_frac_coords = []
batch_all_atom_types = []
batch_frac_coords, batch_num_atoms, batch_atom_types = [], [], []
batch_lengths, batch_angles = [], []

In [13]:
model.to('cuda:0')
print("X")

X


In [14]:
_, _, z = model.encode(batch, xrd_int, xrd_loc, atom_spec)
z = z.to('cuda:0')
batch = batch.to('cuda:0')

In [23]:
def composition_constraint(atom_types, num_atoms, composition_per_atom):
        """
        Restrict the probability distribution from which the atom types are randomly drawn 
        to only include the elements that are present in the crystal.

        atom_types: the atom types in the crystal
        num_atoms: the number of atoms in the crystal
        composition_per_atom: the probability distribution from which the atom types are randomly drawn 

        """

        range_tensor = torch.arange(len(num_atoms))
        range_tensor = range_tensor.to('cuda:0')
        crystal_ids = torch.repeat_interleave(range_tensor, num_atoms)

        # for 1 in 1280
        for index in range(len(composition_per_atom)):
            #set the crystal_index to be the value in the interleaved tensor 
            crystal_index = crystal_ids[index]
            # #get the elements in that crystal and pull out the non-zero values
            elements_in_the_crystal = atom_types[crystal_index][atom_types[crystal_index] != 0]
            mask = torch.zeros_like(composition_per_atom[index], dtype=torch.bool)
            additive_mask_for_normalization = torch.zeros_like(composition_per_atom[index])
            for element_index in elements_in_the_crystal:
                mask[element_index-1]=1
                additive_mask_for_normalization[element_index-1] = 0.0001
            composition_per_atom[index] *= mask
            composition_per_atom[index] += additive_mask_for_normalization


        return composition_per_atom

In [ ]:
def composition_constraint_vectorized(self, atom_types, num_atoms, composition_per_atom):
    """
    Restrict the probability distribution from which the atom types are randomly drawn 
    to only include the elements that are present in the crystal.

    atom_types: the atom types in the crystal
    num_atoms: the number of atoms in the crystal
    composition_per_atom: the probability distribution from which the atom types are randomly drawn 

    """
    # Create a range tensor and repeat it as before
    range_tensor = torch.arange(len(num_atoms), device='cuda:0')
    crystal_ids = torch.repeat_interleave(range_tensor, num_atoms)

    # Convert atom_types into a mask
    atom_mask = atom_types != 0

    # For each unique crystal_id, get its corresponding indices in composition_per_atom
    unique_crystal_ids, counts = torch.unique(crystal_ids, return_counts=True)
    
    start_idx = 0
    for u_id, count in zip(unique_crystal_ids, counts):
        relevant_elements = atom_types[u_id][atom_mask[u_id]]
        mask = torch.zeros_like(composition_per_atom[start_idx], dtype=torch.bool)
        mask[relevant_elements-1] = 1

        # Create additive mask
        additive_mask_for_normalization = mask * 0.0001

        # Apply masks to the relevant segment of composition_per_atom
        composition_per_atom[start_idx:start_idx+count] *= mask
        composition_per_atom[start_idx:start_idx+count] += additive_mask_for_normalization

        # Update start index for next iteration
        start_idx += count

    return composition_per_atom

In [57]:
def decode_stats(z, gt_num_atoms=None, gt_lengths=None, gt_angles=None,
                     teacher_forcing=False, gt_elements = None):
        """
        decode key stats from latent embeddings.
        batch is input during training for teach-forcing.
        """
        if gt_num_atoms is not None:
            num_atoms = model.predict_num_atoms(z)
            lengths_and_angles, lengths, angles = (
                model.predict_lattice(z, gt_num_atoms))
            composition_per_atom = model.predict_composition(z, gt_num_atoms)
            if model.hparams.teacher_forcing_lattice and teacher_forcing:
                lengths = gt_lengths
                angles = gt_angles
        else:
            num_atoms = model.predict_num_atoms(z).argmax(dim=-1)
            lengths_and_angles, lengths, angles = (
                model.predict_lattice(z, num_atoms))
            composition_per_atom = model.predict_composition(z, num_atoms)

        if gt_elements is not None: 
            
            composition_per_atom = composition_constraint_vectorized(gt_elements, gt_num_atoms, composition_per_atom)

        return num_atoms, lengths_and_angles, lengths, angles, composition_per_atom

In [44]:
def composition_constraint_vectorized(atom_types, num_atoms, composition_per_atom):
    """
    Restrict the probability distribution from which the atom types are randomly drawn 
    to only include the elements that are present in the crystal.

    atom_types: the atom types in the crystal
    num_atoms: the number of atoms in the crystal
    composition_per_atom: the probability distribution from which the atom types are randomly drawn 

    """
    # Create a range tensor and repeat it as before
    range_tensor = torch.arange(len(num_atoms), device='cuda:0')
    crystal_ids = torch.repeat_interleave(range_tensor, num_atoms)

    # Convert atom_types into a mask
    atom_mask = atom_types != 0

    # For each unique crystal_id, get its corresponding indices in composition_per_atom
    unique_crystal_ids, counts = torch.unique(crystal_ids, return_counts=True)
    
    start_idx = 0
    for u_id, count in zip(unique_crystal_ids, counts):
        relevant_elements = atom_types[u_id][atom_mask[u_id]]
        mask = torch.zeros_like(composition_per_atom[start_idx], dtype=torch.bool)
        mask[relevant_elements-1] = 1

        # Create additive mask
        additive_mask_for_normalization = mask * 0.0001

        # Apply masks to the relevant segment of composition_per_atom
        composition_per_atom[start_idx:start_idx+count] *= mask
        composition_per_atom[start_idx:start_idx+count] += additive_mask_for_normalization

        # Update start index for next iteration
        start_idx += count

    return composition_per_atom


In [18]:
def build_mlp(in_dim, hidden_dim, fc_num_layers, out_dim):
    mods = [nn.Linear(in_dim, hidden_dim), nn.ReLU()]
    for i in range(fc_num_layers-1):
        mods += [nn.Linear(hidden_dim, hidden_dim), nn.ReLU()]
    mods += [nn.Linear(hidden_dim, out_dim)]
    return nn.Sequential(*mods)

In [19]:
z = z.to('cuda:0')

In [51]:
#test speed
start_time = time.time()
for trial in range(10000):
    (pred_num_atoms, pred_lengths_and_angles, pred_lengths, pred_angles,
        pred_composition_per_atom) = decode_stats(
            z, batch.num_atoms, batch.lengths, batch.angles, True)
# Your code to be timed here
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")

Time taken: 4.579874038696289 seconds


In [21]:
#these are the results without atomic composition constraints 
pred_composition_per_atom[1]

tensor([-27.5840, -34.2045,  -0.2969,   2.2184,  -2.1482, -34.3847,   7.8314,
          7.1494,   5.6262, -31.2265,   3.2186,   3.9847,   1.0280,  -0.1143,
        -33.2018,   0.9749, -32.9094, -38.1796,  -4.7494,  -3.2986,   3.5418,
          5.1475,   4.0626,   3.3409,   5.1851,   4.4533,   4.7122,   5.8970,
          3.0287,   2.3063,   0.3866,  -0.6029,   0.0646, -40.4047, -26.5902,
        -29.0778, -13.8269, -29.9333,  -4.8724,  -1.9282,  -0.1925,   1.7825,
        -33.8307,  -1.5148,   1.2802,   0.7927,  -5.4271,  -6.7305,  -7.2977,
         -3.9417,  -2.7948,   0.0543, -25.6707, -28.2621, -31.0018, -14.7794,
        -11.6086, -32.6720, -24.7264, -32.2830, -34.2241, -32.2905, -27.9115,
        -34.9249, -31.2570, -35.5173, -30.6316, -28.8134, -27.1221, -35.0078,
        -24.7622,  -0.7868,   0.6059,   4.1553,   5.5704,   6.4047,   4.8275,
          4.4218,   3.6351,  -5.0752,  -6.9301,  -2.9247,  -3.3073, -28.8674,
        -32.2481, -28.9924, -33.7839, -28.2463, -38.1396, -30.81

In [34]:
atom_spec = atom_spec.to('cuda:0')

In [60]:
#test speed
start_time = time.time()
for trial in range(20):
    (pred_num_atoms, pred_lengths_and_angles, pred_lengths, pred_angles,
            pred_composition_per_atom) = decode_stats(
                z, batch.num_atoms, batch.lengths, batch.angles, True, gt_elements = atom_spec)
# Your code to be timed here
end_time = time.time()
print(f"Time taken: {end_time - start_time} seconds")

Time taken: 1.3254468441009521 seconds


In [48]:
pred_composition_per_atom[1]

tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 7.8315, 7.1495, 5.6263,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 5.1476, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 6.4048, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000], device='cuda:0', grad_fn=<SelectBackward>)

In [ ]:
(pred_num_atoms, pred_lengths_and_angles, pred_lengths, pred_angles,
        pred_composition_per_atom) = decode_stats(
            z, batch.num_atoms, batch.lengths, batch.angles, True, gt_elements = atom_spec)

In [70]:
noise_level = torch.randint(0, model.sigmas.size(0),
                                    (batch.num_atoms.size(0),),
                                    device=model.device)

In [71]:
used_sigmas_per_atom = model.sigmas[noise_level].repeat_interleave(
            batch.num_atoms, dim=0)

type_noise_level = torch.randint(0, model.type_sigmas.size(0),
                                (batch.num_atoms.size(0),),
                                device=model.device)
used_type_sigmas_per_atom = (
    model.type_sigmas[type_noise_level].repeat_interleave(
        batch.num_atoms, dim=0))

In [72]:
pred_composition_probs = F.softmax(
    pred_composition_per_atom.detach(), dim=-1)
atom_type_probs = (
    F.one_hot(batch.atom_types - 1, num_classes=MAX_ATOMIC_NUM) +
    pred_composition_probs * used_type_sigmas_per_atom[:, None])


In [75]:
atom_type_probs = composition_constraint_vectorized(batch.atom_types - 1, batch.num_atoms, atom_type_probs)

In [76]:
rand_atom_types = torch.multinomial(
            atom_type_probs, num_samples=1).squeeze(1) + 1
cart_noises_per_atom = (
            torch.randn_like(batch.frac_coords) *
            used_sigmas_per_atom[:, None])
cart_coords = frac_to_cart_coords(
    batch.frac_coords, pred_lengths, pred_angles, batch.num_atoms)
cart_coords = cart_coords + cart_noises_per_atom
noisy_frac_coords = cart_to_frac_coords(
    cart_coords, pred_lengths, pred_angles, batch.num_atoms)

In [77]:
noisy_frac_coords = noisy_frac_coords.to('cuda:0')
rand_atom_types = rand_atom_types.to('cuda:0')
pred_lengths = pred_lengths.to('cuda:0')
pred_angles = pred_angles.to('cuda:0')

In [78]:
pred_cart_coord_diff, pred_atom_types = model.decoder(
    z, noisy_frac_coords, rand_atom_types, batch.num_atoms, pred_lengths, pred_angles)

In [79]:
ld_kwargs = SimpleNamespace(n_step_each=100,
                            step_lr=1e-4,
                            min_sigma=0,
                            save_traj=True,
                            disable_bar=False)

In [80]:
gt_num_atoms = batch.num_atoms
gt_atom_types = None
gt_num_atoms = gt_num_atoms.cpu()
z = z.cpu()
tsach_atom_types = batch.atom_types
tsach_atom_types = tsach_atom_types.cpu()

In [81]:
model = model.cuda()

In [91]:
def langevin_dynamics(z, ld_kwargs, gt_num_atoms=None, gt_atom_types=None, tsach_atom_types=None):
    """
    decode crystral structure from latent embeddings.
    ld_kwargs: args for doing annealed langevin dynamics sampling:
        n_step_each:  number of steps for each sigma level.
        step_lr:      step size param.
        min_sigma:    minimum sigma to use in annealed langevin dynamics.
        save_traj:    if <True>, save the entire LD trajectory.
        disable_bar:  disable the progress bar of langevin dynamics.
    gt_num_atoms: if not <None>, use the ground truth number of atoms.
    gt_atom_types: if not <None>, use the ground truth atom types.
    """

    if ld_kwargs.save_traj:
        all_frac_coords = []
        all_pred_cart_coord_diff = []
        all_noise_cart = []
        all_atom_types = []

    # obtain key stats.
    num_atoms, _, lengths, angles, composition_per_atom = model.decode_stats(
        z, gt_num_atoms)
    if gt_num_atoms is not None:
        num_atoms = gt_num_atoms

    composition_per_atom = F.softmax(composition_per_atom, dim=-1)

    composition_per_atom = composition_per_atom.cuda()
    num_atoms = num_atoms.cuda()
    if gt_atom_types is None:
        cur_atom_types = model.sample_composition(
            composition_per_atom, num_atoms)
    else:
        cur_atom_types = gt_atom_types

    # init coords.
    cur_frac_coords = torch.rand((num_atoms.sum(), 3), device=z.device)

    # annealed langevin dynamics.
    for sigma in tqdm(model.sigmas, total=model.sigmas.size(0), disable=ld_kwargs.disable_bar):
        if sigma < ld_kwargs.min_sigma:
            break
        step_size = ld_kwargs.step_lr * (sigma / model.sigmas[-1]) ** 2

        for step in range(ld_kwargs.n_step_each):
            noise_cart = torch.randn_like(
                cur_frac_coords) * torch.sqrt(step_size * 2)
            pred_cart_coord_diff, pred_atom_types = model.decoder(
                z, cur_frac_coords, cur_atom_types, num_atoms, lengths, angles)
            cur_cart_coords = frac_to_cart_coords(
                cur_frac_coords, lengths, angles, num_atoms)
            pred_cart_coord_diff = pred_cart_coord_diff / sigma
            cur_cart_coords = cur_cart_coords + step_size * pred_cart_coord_diff + noise_cart
            cur_frac_coords = cart_to_frac_coords(
                cur_cart_coords, lengths, angles, num_atoms)

            if gt_atom_types is None:
                cur_atom_types = torch.argmax(pred_atom_types, dim=1) + 1

            if ld_kwargs.save_traj:
                all_frac_coords.append(cur_frac_coords)
                all_pred_cart_coord_diff.append(
                    step_size * pred_cart_coord_diff)
                all_noise_cart.append(noise_cart)
                all_atom_types.append(cur_atom_types)


    output_dict = {'num_atoms': num_atoms, 'lengths': lengths, 'angles': angles,
                'frac_coords': cur_frac_coords, 'atom_types': cur_atom_types,
                'is_traj': False}

    if ld_kwargs.save_traj:
        output_dict.update(dict(
            all_frac_coords=torch.stack(all_frac_coords, dim=0),
            all_atom_types=torch.stack(all_atom_types, dim=0),
            all_pred_cart_coord_diff=torch.stack(
                all_pred_cart_coord_diff, dim=0),
            all_noise_cart=torch.stack(all_noise_cart, dim=0),
            is_traj=True))

    return output_dict


In [83]:
z = z.cuda()
gt_num_atoms = gt_num_atoms.cuda()

In [84]:
z = z[[0]]

In [85]:
z

tensor([[ 21.9119,  31.1830,  38.4384,  44.6807,  50.2980,  55.4897,  65.0345,
          69.5230,  73.8836,  78.1501,  82.3508,  86.5104,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
           0.0000,   0.0000,   0.0000,   0.0000,   0

In [86]:
gt_num_atoms = batch.num_atoms[[0]]
gt_num_atoms
gt_atom_types = None

In [87]:
def composition_constraint(atom_types, num_atoms, composition_per_atom):
    """
    Restrict the probability distribution from which the atom types are randomly drawn 
    to only include the elements that are present in the crystal.

    atom_types: the atom types in the crystal
    num_atoms: the number of atoms in the crystal
    composition_per_atom: the probability distribution from which the atom types are randomly drawn 

    """

    range_tensor = torch.arange(len(num_atoms))
    range_tensor = range_tensor.cpu()

    crystal_ids = torch.repeat_interleave(range_tensor, num_atoms)
    unique_crystal_ids = crystal_ids.unique()
    crystal_to_atom_types = {crystal_id.item(): atom_types[crystal_ids == crystal_id].unique() for crystal_id in unique_crystal_ids}

    mask = torch.zeros_like(composition_per_atom, dtype=torch.bool)
    mask = mask.cuda(0)

    for i, (atom_type, crystal_id) in enumerate(zip(atom_types, crystal_ids)):

        crystal_id = crystal_id.item()
        atom_types_in_crystal = crystal_to_atom_types[crystal_id]
        atom_types_in_crystal = atom_types_in_crystal.to(dtype=torch.long)
        mask[i, atom_types_in_crystal] = 1

    epsilon = 1e-10
    composition_per_atom = composition_per_atom + epsilon # to avoid dividing by 0 
    composition_per_atom = composition_per_atom * mask.float()

    return composition_per_atom

In [88]:
model = model.cuda()

In [89]:
tsach_atom_types = batch.atom_types

In [92]:
outputs = langevin_dynamics(z, ld_kwargs, gt_num_atoms, gt_atom_types, tsach_atom_types)

 40%|████      | 20/50 [00:46<01:09,  2.31s/it]


RuntimeError: CUDA out of memory. Tried to allocate 2.00 MiB (GPU 0; 31.74 GiB total capacity; 30.02 GiB already allocated; 2.88 MiB free; 30.44 GiB reserved in total by PyTorch)

In [93]:
batch

Batch(edge_index=[2, 8088], y=[256, 1], frac_coords=[1280, 3], atom_types=[1280], lengths=[256, 3], angles=[256, 3], to_jimages=[8088, 3], num_atoms=[256], num_bonds=[256], num_nodes=1280, batch=[1280], ptr=[257])

In [94]:
num_atoms, _, lengths, angles, composition_per_atom = model.decode_stats(z, gt_num_atoms)

In [95]:
print(torch.max(composition_per_atom, dim = 1))
print(num_atoms)
print(lengths)
print(angles)

torch.return_types.max(
values=tensor([7.8314, 7.8314, 7.8314, 7.8314, 7.8314], device='cuda:0',
       grad_fn=<MaxBackward0>),
indices=tensor([6, 6, 6, 6, 6], device='cuda:0'))
tensor([[ -3.3285,  -7.4109,  -3.8316, -11.4725,  -7.6950,  41.8840, -11.9111,
         -14.8411,  -7.4835,  -6.5635,  -5.6949,  -5.2582, -18.6326, -16.0603,
          -0.9327,  -5.4197,  -6.8195,  -4.8923,  -8.9087,  -9.0741,  -9.8191]],
       device='cuda:0', grad_fn=<AddmmBackward>)
tensor([[4.0471, 4.0456, 4.0509]], device='cuda:0')
tensor([[90., 90., 90.]], device='cuda:0')


In [53]:
outputs

{'num_atoms': tensor([5], device='cuda:0'),
 'lengths': tensor([[8.5954, 8.5954, 8.5954]], device='cuda:0'),
 'angles': tensor([[90., 90., 90.]], device='cuda:0'),
 'frac_coords': tensor([[0.1144, 0.4182, 0.8519],
         [0.1083, 0.6902, 0.3693],
         [0.9534, 0.2090, 0.3743],
         [0.4383, 0.0974, 0.5632],
         [0.6173, 0.5898, 0.6424]], device='cuda:0',
        grad_fn=<RemainderBackward0>),
 'atom_types': tensor([13, 13, 13, 13, 13], device='cuda:0'),
 'is_traj': False}

In [54]:
print(batch.num_atoms[[1]])
print(batch.atom_types[[np.arange(5, 10)]])
print(batch.frac_coords[[np.arange(5,10)]])
print(batch.lengths[[1]])
print(batch.angles[[1]])

tensor([5], device='cuda:0')
tensor([81, 44,  8,  8,  9], device='cuda:0')
tensor([[5.0577e-01, 5.0000e-01, 5.0000e-01],
        [3.3725e-04, 0.0000e+00, 0.0000e+00],
        [4.9976e-01, 5.0000e-01, 0.0000e+00],
        [4.7342e-03, 5.0000e-01, 5.0000e-01],
        [4.9947e-01, 0.0000e+00, 5.0000e-01]], device='cuda:0')
tensor([[4.3490, 4.3490, 4.3490]], device='cuda:0')
tensor([[90., 90., 90.]], device='cuda:0')


In [96]:
torch.cuda.empty_cache()

In [97]:
def estimate_target_coords(pred_cart_coord_diff, noisy_cart_coords, used_sigmas_per_atom):
    """
    Estimate the target Cartesian coordinates.

    Parameters:
    pred_cart_coord_diff (Tensor): The predicted differences in Cartesian coordinates.
    noisy_cart_coords (Tensor): The noisy Cartesian coordinates.
    used_sigmas_per_atom (Tensor): Used sigmas per atom for normalization.

    Returns:
    Tensor: The estimated target Cartesian coordinates.
    """

    # Reverse the normalization applied during the loss calculation
    adjusted_pred_diff = pred_cart_coord_diff * used_sigmas_per_atom[:, None]

    # Apply the predicted coordinate differences to the noisy coordinates
    estimated_target_coords = noisy_cart_coords + adjusted_pred_diff

    return estimated_target_coords

In [98]:
pred_cart_cords = estimate_target_coords(pred_cart_coord_diff, cart_coords, used_sigmas_per_atom)
pred_frac_coords = cart_to_frac_coords(pred_cart_cords, pred_lengths, pred_angles, batch.num_atoms)

In [99]:
def diffraction_pattern_post_processing(xrd_loc, xrd_int):
    #get the xrd pattern
    adjusted_vector = np.zeros(256)
    minimum = min(256, len(xrd_loc))
    adjusted_vector[:minimum] = xrd_loc[:minimum]

    #get the xrd intensities
    adjusted_intensities = np.zeros(256)
    minimum = min(256, len(xrd_int))
    adjusted_intensities[:minimum] = xrd_int[:minimum]

    return adjusted_vector, adjusted_intensities

In [100]:
def get_diffraction_pattern(pred_num_atoms, pred_frac_coords, pred_atom_types, pred_lengths, pred_angles):
        """
        inputs: 
        pred_num_atoms: the predicted number of atoms in the crystal
        pred_frac_coords: the predicted fractional coordinates of the atoms in the crystal
        pred_atom_types: the predicted atom types in the crystal
        pred_lengths: the predicted lengths of the crystal
        pred_angles: the predicted angles of the crystal

        """
        peak_locations = []
        peak_intensities = []
        counter = 0 
        for i in range(len(pred_num_atoms)):
            def tensor_to_list(tensor_data):
                # If the data is a tensor, convert to list
                if isinstance(tensor_data, torch.Tensor):
                    return tensor_data.tolist()
                return tensor_data

            num_atoms = pred_num_atoms[i].item()
            atom_types = tensor_to_list(pred_atom_types[counter:counter+num_atoms])
            frac_coords = tensor_to_list(pred_frac_coords[counter:counter+num_atoms])
            angles = tensor_to_list(pred_angles[i])
            lengths = tensor_to_list(pred_lengths[i])
        
           
            atomic_species = [Element.from_Z(atom_type).symbol for atom_type in atom_types]
            
            atomic_species = tensor_to_list(atom_types)
  
            frac_coords = tensor_to_list(frac_coords)
            alpha, beta, gamma = tensor_to_list(angles)
            a, b, c = tensor_to_list(lengths)

            lattice = Lattice.from_parameters(a=a, b=b, c=c, alpha=alpha, beta=beta, gamma=gamma)
            structure = Structure(lattice, species=atomic_species, coords=frac_coords, coords_are_cartesian=False)

            pattern = xrd_calculator.get_pattern(structure)

            #use diffraction pattern post processing to get the diffraction pattern
            adjusted_vector, adjusted_intensities = diffraction_pattern_post_processing(pattern.x, pattern.y)

            peak_locations.append(adjusted_vector)
            peak_intensities.append(adjusted_intensities)
            
            counter = counter+num_atoms
        
        peak_locations = torch.tensor(peak_locations).to('cuda:0').to(torch.float32)        
        peak_intensities = torch.tensor(peak_intensities).to('cuda:0').to(torch.float32)

        return peak_locations, peak_intensities

In [63]:
batch

Batch(edge_index=[2, 8088], y=[256, 1], frac_coords=[1280, 3], atom_types=[1280], lengths=[256, 3], angles=[256, 3], to_jimages=[8088, 3], num_atoms=[256], num_bonds=[256], num_nodes=1280, batch=[1280], ptr=[257])

In [64]:
decoded_xrd_loc, decoded_xrd_int = get_diffraction_pattern(batch.num_atoms, batch.frac_coords, batch.atom_types, batch.lengths, batch.angles)

In [69]:
batch.atom_types[0:5]

tensor([22, 76,  7,  8,  9], device='cuda:0')

In [68]:
atom_spec[0]

tensor([ 9,  8, 76, 22,  7,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0, 

In [66]:
decoded_xrd_int[0]

tensor([100.0000,  89.2229,  40.5148,  33.3204,  57.6611,  30.7869,  16.2317,
         29.9895,  19.5877,  12.8611,   4.1240,  10.7317,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.00

In [65]:
xrd_int[0]

tensor([100.0000,  89.2229,  40.5148,  33.3204,  57.6611,  30.7869,  16.2317,
         29.9895,  19.5877,  12.8611,   4.1240,  10.7317,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.0000,
          0.0000,   0.0000,   0.0000,   0.0000,   0.0000,   0.00

In [69]:
torch.set_printoptions(threshold=500000) 

In [101]:
scores, most_likely_atoms = torch.max(pred_atom_types, dim=1)
most_likely_atoms = most_likely_atoms + 1

In [102]:
decoded_xrd_loc, decoded_xrd_int = get_diffraction_pattern(batch.num_atoms, pred_frac_coords, most_likely_atoms, pred_lengths, pred_angles)

RuntimeError: CUDA out of memory. Tried to allocate 2.00 MiB (GPU 0; 31.74 GiB total capacity; 30.02 GiB already allocated; 2.88 MiB free; 30.44 GiB reserved in total by PyTorch)

In [72]:
num_atom_loss = model.num_atom_loss(pred_num_atoms, batch)
lattice_loss = model.lattice_loss(pred_lengths_and_angles, batch)
composition_loss = model.composition_loss(
    pred_composition_per_atom, batch.atom_types, batch)
coord_loss = model.coord_loss(
    pred_cart_coord_diff, noisy_frac_coords, used_sigmas_per_atom, batch)
type_loss = model.type_loss(pred_atom_types, batch.atom_types,
                        used_type_sigmas_per_atom, batch)

/home/gridsan/tmackey/cdvae/cdvae/common/data_utils.py:621: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float)


In [73]:
print(num_atom_loss)
print(lattice_loss)
print(composition_loss)
print(coord_loss)
print(type_loss)

tensor(0., device='cuda:0', grad_fn=<NllLossBackward>)
tensor(3.0697, device='cuda:0', grad_fn=<MseLossBackward>)
tensor(76.6060, device='cuda:0', grad_fn=<MeanBackward0>)
tensor(1.5311, device='cuda:0', grad_fn=<MeanBackward0>)
tensor(197.5703, device='cuda:0', grad_fn=<MeanBackward0>)


In [51]:
def normalize_tensor(tensor, min_val, max_val):
    """Normalize the tensor to the range [min_val, max_val]."""
    tensor_min = tensor.min()
    tensor_max = tensor.max()
    # Normalize to [0, 1]
    tensor = (tensor - tensor_min) / (tensor_max - tensor_min)
    # Scale to [min_val, max_val]
    tensor = tensor * (max_val - min_val) + min_val
    return tensor


In [52]:
def diffraction_pattern_loss(decoded_xrd, decoded_int, gt_xrd_loc, gt_xrd_int):
    #get the rmse loss between the decoded xrd and the gt xrd loc
    xrd_loc_mse_loss = F.mse_loss(decoded_xrd, gt_xrd_loc)
    
    decoded_int_norm = normalize_tensor(decoded_int, 0, 100)
    gt_int_norm =normalize_tensor(gt_xrd_int, 0, 100)
    

    #get the mean cosine similarity between the decoded xrd and the gt xrd loc
    xrd_int_mse_loss = F.mse_loss(decoded_int_norm, gt_int_norm)
    
    xrd_int_mse_loss

    return xrd_loc_mse_loss, xrd_int_mse_loss


In [53]:
import matplotlib.pyplot as plt

def plot_histogram(tensor):
    # Convert the tensor to a 1D array
    values = tensor.view(-1).numpy()  # If you're using PyTorch
    values = values[values != 0]  
    # Create the histogram
    plt.figure(figsize=(10, 6))
    plt.hist(values, bins=500, color='blue', alpha=0.7)  # You can change the number of bins
    plt.title('Histogram of Values in Tensor')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.grid(True)
    plt.show()

# Plot histogram
# plot_histogram(xrd_loc)
# print(torch.max(xrd_loc))

some interesting plots

In [54]:
# Plot histogram
# plot_histogram(xrd_int)
# plot_histogram(decoded_xrd_int)

In [56]:
decoded_xrd_loc = decoded_xrd_loc.cpu()
# print(decoded_xrd_loc[3])
decoded_xrd_int = decoded_xrd_int.cpu()
# print(decoded_xrd_int[3])
xrd_loc= xrd_loc.cpu()
# print(decoded_xrd_loc[3])
xrd_int = xrd_int.cpu()
# print(xrd_int[3])

In [57]:
decoded_xrd_loc.requires_grad_(True)
decoded_xrd_int.requires_grad_(True)
xrd_loc.requires_grad_(True)
xrd_int.requires_grad_(True)
loc_loss, int_loss = diffraction_pattern_loss(decoded_xrd_loc, decoded_xrd_int, xrd_loc, xrd_int)

In [59]:
print(loc_loss)
print(int_loss)

tensor(0.1253, grad_fn=<MseLossBackward>)
tensor(24.7016, grad_fn=<MseLossBackward>)


In [319]:
pred_loc = model.fc_diffraction_pattern(z)

In [320]:
loc_loss, int_loss = diffraction_pattern_loss(pred_loc, decoded_xrd_int, xrd_loc, xrd_int)

In [321]:
print(loc_loss)

tensor(156.1326, grad_fn=<MseLossBackward>)


In [322]:
print(loc_loss)
print(int_loss)

tensor(156.1326, grad_fn=<MseLossBackward>)
tensor(44.2598, grad_fn=<MseLossBackward>)


In [331]:
optimization = True

if optimization:
    z = z.detach().clone().requires_grad_()
    num_gradient_steps=50000
    lr=1e-3
    opt = Adam([z], lr=lr)
    model.freeze()
    interval = num_gradient_steps // 10
    for i in tqdm(range(num_gradient_steps)):
        opt.zero_grad()
        
        (pred_num_atoms, pred_lengths_and_angles, pred_lengths, pred_angles,
        pred_composition_per_atom) = model.decode_stats(
            z, batch.num_atoms, batch.lengths, batch.angles, True)
        noise_level = torch.randint(0, model.sigmas.size(0),
                                    (batch.num_atoms.size(0),),
                                    device=model.device)
        used_sigmas_per_atom = model.sigmas[noise_level].repeat_interleave(
            batch.num_atoms, dim=0)

        type_noise_level = torch.randint(0, model.type_sigmas.size(0),
                                        (batch.num_atoms.size(0),),
                                        device=model.device)
        used_type_sigmas_per_atom = (
            model.type_sigmas[type_noise_level].repeat_interleave(
                batch.num_atoms, dim=0))
        pred_composition_probs = F.softmax(pred_composition_per_atom.detach(), dim=-1)
        atom_type_probs = (F.one_hot(batch.atom_types - 1, num_classes=MAX_ATOMIC_NUM) + pred_composition_probs * used_type_sigmas_per_atom[:, None])
        atom_type_probs = composition_constraint(batch.atom_types - 1, batch.num_atoms, atom_type_probs)
        rand_atom_types = torch.multinomial(atom_type_probs, num_samples=1).squeeze(1) + 1
        cart_noises_per_atom = (torch.randn_like(batch.frac_coords) *used_sigmas_per_atom[:, None])
        cart_coords = frac_to_cart_coords(batch.frac_coords, pred_lengths, pred_angles, batch.num_atoms)
        cart_coords = cart_coords + cart_noises_per_atom
        noisy_frac_coords = cart_to_frac_coords(cart_coords, pred_lengths, pred_angles, batch.num_atoms)
        pred_cart_coord_diff, pred_atom_types = model.decoder(z, noisy_frac_coords, rand_atom_types, batch.num_atoms, pred_lengths, pred_angles)
        pred_cart_cords = estimate_target_coords(pred_cart_coord_diff, cart_coords, used_sigmas_per_atom)
        pred_frac_coords = cart_to_frac_coords(pred_cart_cords, pred_lengths, pred_angles, batch.num_atoms)
        scores, most_likely_atoms = torch.max(pred_atom_types, dim=1)
        most_likely_atoms = most_likely_atoms + 1
        decoded_xrd_loc, decoded_xrd_int = get_diffraction_pattern(batch.num_atoms, pred_frac_coords, most_likely_atoms, pred_lengths, pred_angles)
        num_atom_loss = model.num_atom_loss(pred_num_atoms, batch)
        lattice_loss = model.lattice_loss(pred_lengths_and_angles, batch)
        composition_loss = model.composition_loss(
            pred_composition_per_atom, batch.atom_types, batch)
        coord_loss = model.coord_loss(
            pred_cart_coord_diff, noisy_frac_coords, used_sigmas_per_atom, batch)
        type_loss = model.type_loss(pred_atom_types, batch.atom_types,
                                used_type_sigmas_per_atom, batch)
        
        decoded_xrd_loc = decoded_xrd_loc.cpu()
        decoded_xrd_int = decoded_xrd_int.cpu()
        xrd_loc= xrd_loc.cpu()
        xrd_int = xrd_int.cpu()
        loss = F.mse_loss(decoded_xrd_int, xrd_int)
        if i % 1 == 0 or i == (num_gradient_steps-1):
            print(num_atom_loss)
            print(lattice_loss)
            print(composition_loss)
            print(coord_loss)
            print(type_loss)
            print("diff loss: ", loss)

  0%|          | 1/50000 [00:03<52:45:17,  3.80s/it]

tensor(0.7953, grad_fn=<NllLossBackward>)
tensor(2.3508, grad_fn=<MseLossBackward>)
tensor(3.4449, grad_fn=<MeanBackward0>)
tensor(1.1567, grad_fn=<MeanBackward0>)
tensor(45.4812, grad_fn=<MeanBackward0>)
diff loss:  tensor(39.5375, grad_fn=<MseLossBackward>)


  0%|          | 2/50000 [00:08<57:27:54,  4.14s/it]

tensor(0.7953, grad_fn=<NllLossBackward>)
tensor(2.3508, grad_fn=<MseLossBackward>)
tensor(3.4449, grad_fn=<MeanBackward0>)
tensor(1.1978, grad_fn=<MeanBackward0>)
tensor(56.6437, grad_fn=<MeanBackward0>)
diff loss:  tensor(26.3628, grad_fn=<MseLossBackward>)


  0%|          | 3/50000 [00:12<55:52:48,  4.02s/it]

...

tensor(0.7953, grad_fn=<NllLossBackward>)
tensor(2.3508, grad_fn=<MseLossBackward>)
tensor(3.4449, grad_fn=<MeanBackward0>)
tensor(1.1374, grad_fn=<MeanBackward0>)
tensor(62.7233, grad_fn=<MeanBackward0>)
diff loss:  tensor(33.1961, grad_fn=<MseLossBackward>)


  0%|          | 61/50000 [03:53<53:16:58,  3.84s/it]

tensor(0.7953, grad_fn=<NllLossBackward>)
tensor(2.3508, grad_fn=<MseLossBackward>)
tensor(3.4449, grad_fn=<MeanBackward0>)
tensor(1.1415, grad_fn=<MeanBackward0>)
tensor(54.6316, grad_fn=<MeanBackward0>)
diff loss:  tensor(35.1954, grad_fn=<MseLossBackward>)


  0%|          | 61/50000 [03:55<53:35:50,  3.86s/it]


KeyboardInterrupt: 

In [ ]:
force_num_atoms = True
gt_num_atoms = batch.num_atoms if force_num_atoms else None
force_atom_types = False
gt_atom_types = batch.atom_types if force_atom_types else None


In [ ]:
ld_kwargs = SimpleNamespace(n_step_each=100,
                        step_lr=1e-4,
                        min_sigma=0,
                        save_traj=False,
                        disable_bar=False)

In [ ]:
#put everything on the gpu 
z = z.cuda()
gt_num_atoms = gt_num_atoms.cuda()
gt_atom_types = gt_atom_types.cuda()

In [ ]:
outputs = model.langevin_dynamics(
                    z, ld_kwargs, gt_num_atoms, gt_atom_types, tsach_atom_types=batch.atom_types)